# Model Training

1. **Travel time**: distance_km, hour, day_of_week → duration_min  
2. **Pricing**: distance, weight, volume, urgency, hour, day, demand → price  
3. **Demand forecast**: hour, day_of_week → request count  
4. **Hotspots**: K-Means on pickup/drop locations for agent pre-positioning

## 1. Travel Time Model

In [1]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import lightgbm as lgb

df = pd.read_csv("../data/raw/travel_times.csv")
# Add rush hour flag (7-9am, 5-8pm)
df["is_rush"] = ((df["hour"] >= 7) & (df["hour"] <= 9)) | ((df["hour"] >= 17) & (df["hour"] <= 20))
df["is_rush"] = df["is_rush"].astype(int)
X = df[["distance_km", "hour", "day_of_week", "is_rush"]]
y = df["duration_min"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = lgb.LGBMRegressor(n_estimators=200, num_leaves=31, learning_rate=0.08, max_depth=6,
                          min_child_samples=20, random_state=42, verbosity=-1)
model.fit(X_train, y_train)

Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(model, "../models/travel_time_model.joblib")
print(f"Val MAE: {(model.predict(X_val) - y_val).abs().mean():.2f} min")

Val MAE: 0.75 min


## 2. Pricing Model

In [2]:
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/raw/pricing_synthetic.csv")
le = LabelEncoder()
df["urgency_enc"] = le.fit_transform(df["urgency"])

X = df[["distance_km", "weight_kg", "volume_l", "urgency_enc", "hour", "day_of_week", "demand_score"]]
y = df["price"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = lgb.LGBMRegressor(n_estimators=150, num_leaves=31, learning_rate=0.08, max_depth=7,
                          min_child_samples=15, random_state=42, verbosity=-1)
model.fit(X_train, y_train)

joblib.dump(model, "../models/pricing_model.joblib")
joblib.dump({"encoder": le, "urgency_classes": list(le.classes_)}, "../models/urgency_encoder.joblib")
print(f"Val MAE: {(model.predict(X_val) - y_val).abs().mean():.2f} INR")

Val MAE: 16.32 INR


## 3. Demand Forecasting

In [3]:
# City-wide demand: (hour, dow) -> count
df = pd.read_csv("../data/raw/travel_times.csv")
city_demand = df.groupby(["hour", "day_of_week"]).size().reset_index(name="count")
# Cyclic encoding for hour (6am vs 6pm are different)
city_demand["hour_sin"] = np.sin(2 * np.pi * city_demand["hour"] / 24)
city_demand["hour_cos"] = np.cos(2 * np.pi * city_demand["hour"] / 24)
city_demand["is_rush"] = ((city_demand["hour"] >= 7) & (city_demand["hour"] <= 9)) | ((city_demand["hour"] >= 17) & (city_demand["hour"] <= 20))
city_demand["is_rush"] = city_demand["is_rush"].astype(int)
X_d = city_demand[["hour", "day_of_week", "hour_sin", "hour_cos", "is_rush"]]
y_d = city_demand["count"]
model_demand = lgb.LGBMRegressor(n_estimators=80, max_depth=4, learning_rate=0.15, random_state=42, verbosity=-1).fit(X_d, y_d)
joblib.dump(model_demand, "../models/demand_model.joblib")
print("Demand model saved (LightGBM + cyclic hour)")

Demand model saved (LightGBM + cyclic hour)


## 4. Hotspots (for agent pre-positioning)

In [4]:
from sklearn.cluster import KMeans
df_ht = pd.read_csv("../data/raw/travel_times.csv")
points = pd.concat([
    df_ht[["origin_lat", "origin_lng"]].rename(columns={"origin_lat":"lat","origin_lng":"lng"}),
    df_ht[["dest_lat", "dest_lng"]].rename(columns={"dest_lat":"lat","dest_lng":"lng"})
])
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10).fit(points[["lat","lng"]])
hotspots = [{"lat": round(c[0], 4), "lng": round(c[1], 4), "rank": i+1} 
            for i, c in enumerate(kmeans.cluster_centers_)]
joblib.dump(hotspots, "../models/hotspots.joblib")
print(f"Saved {len(hotspots)} hotspots")

Saved 10 hotspots
